# SAE known-entity feature — correcting a hallucination

Replicates the steering experiment from **"Do I Know This Entity? Knowledge Awareness and Hallucinations in Language Models"** ([arXiv:2411.14257](https://arxiv.org/abs/2411.14257)) on Gemma-2-9B-it.

The question smuggles in a false premise (LeBron James won his first MVP in 2009, not 2006), and the baseline hallucinates: it accepts the wrong year and answers as if the premise were true. A "known entity" direction, read directly from the Gemma Scope SAE decoder (layer 31, width 16k, feature 88) and added at the last prompt position, restores the model's knowledge awareness: at moderate scale it corrects the year, and at high scale it rejects the false premise outright.

**Execution note:** Outputs are cleared after the API migration. Run the notebook from top to bottom to obtain results for your environment.


In [ ]:
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")  # quiet engine boot logs

from vllm import LLM, SamplingParams
import easysteer.vectors as vec
from vllm.steer_vectors import ApplySpec, SteeringSpec, VectorSpec

MODEL = os.environ.get("EASYSTEER_MODEL", "google/gemma-2-9b-it")  # google/gemma-2-9b-it

llm = LLM(
    model=MODEL,
    tensor_parallel_size=int(os.environ.get("EASYSTEER_TP", "1")),
    enable_steer_vector=True,
    steer_algorithms=["direct"],
)
tokenizer = llm.get_tokenizer()

In [ ]:
# False premise: LeBron James won his first MVP in 2009, not 2006.
messages = [
    {"role": "user", "content": "Who was the head coach of the Cleveland Cavaliers when LeBron James won his first MVP in 2006?"},
]
prompt_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, return_dict=False, add_generation_prompt=True,
)
prompt = {"prompt_token_ids": prompt_ids}
params = SamplingParams(temperature=0, max_tokens=256, skip_special_tokens=False)

baseline = llm.generate(prompt, params, use_tqdm=False, steering=False)
print("=====Baseline=====")
print(baseline[0].outputs[0].text)

In [ ]:
import numpy as np
import torch

# Gemma Scope SAE for the layer-31 residual stream (width 16k).
from huggingface_hub import hf_hub_download

SAE_PARAMS = os.environ.get("EASYSTEER_SAE_PARAMS") or hf_hub_download(
    repo_id="google/gemma-scope-9b-it-res",
    filename="layer_31/width_16k/average_l0_76/params.npz",
)

data = np.load(SAE_PARAMS)
W_dec = data["W_dec"]  # (16384, 3584): one decoder row per SAE feature
# The paper identifies features 88 and 5038 as the strongest
# "known entity" directions.
feature = 88  # also try 5038
torch.save(W_dec[feature, :], "james.pt")

In [ ]:
# Larger scale pushes harder toward "known entity": α=500 corrects
# the year, α=2000 rejects the false premise outright.
for scale in (500, 2000):
    steering = SteeringSpec(vectors=[
        VectorSpec(
            data=vec.from_pt_direction("james.pt", layers=[31]),
            scale=scale,
            layers=[31],
            apply=ApplySpec(prompt_positions=[-1]),
        ),
    ])
    steered = llm.generate(prompt, params, steering=steering, use_tqdm=False)
    print(f"=====α {scale}=====")
    print(steered[0].outputs[0].text)